## Imports & Setup

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import matplotlib.pyplot as plt
import json
from pathlib import Path

from app.models.transformer import BehavioralTransformer, TransformerConfig
from app.models.gan_autoencoder import GANAutoencoderKYC
from app.models.ensemble import ModelEnsemble
from synthetic_data_generator.behavioral import (
    UserProfileGenerator, TransactionSequenceGenerator, AnomalyInjector
)
from synthetic_data_generator.kyc import DocumentMetadataGenerator, ForgerySimulator

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT_DIR = Path("../checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)
print(f"Training on: {DEVICE}")


## Generate / Load Behavioral Data

In [ ]:
print("Generating behavioral data...")
profiles = UserProfileGenerator(seed=42).generate_batch(n_users=500, n_fraud_users=50)
seq_gen  = TransactionSequenceGenerator(seed=42)
injector = AnomalyInjector(seed=42)

sequences, labels = [], []
for profile in profiles:
    seq = seq_gen.generate_for_user(profile, n_transactions=50)
    label = 0
    if np.random.random() < 0.15:
        seq = injector.inject(seq, profile)
        label = 1

    # Encode to (50, 32) — reuse the same encoder as sequential_service
    from app.services.sequential_service import _transaction_to_vector
    vectors = [_transaction_to_vector(t) for t in seq[-50:]]
    while len(vectors) < 50:
        vectors.insert(0, np.zeros(32, dtype=np.float32))
    sequences.append(np.stack(vectors, axis=0))
    labels.append(label)

X = torch.tensor(np.array(sequences), dtype=torch.float32)
y = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)

print(f"Dataset: {len(X)} sequences | Fraud rate: {y.mean().item():.2%}")


## Train/Val Split & DataLoaders

In [ ]:
dataset = TensorDataset(X, y)
n_train = int(0.8 * len(dataset))
n_val   = len(dataset) - n_train
train_ds, val_ds = random_split(dataset, [n_train, n_val])

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False)

print(f"Train: {n_train} | Val: {n_val}")


## Train Behavioral Transformer

In [ ]:
config = TransformerConfig()
model  = BehavioralTransformer(config).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

# Class-weighted BCE for imbalanced data
pos_weight = torch.tensor([(y == 0).sum() / (y == 1).sum()]).to(DEVICE)
criterion  = nn.BCELoss()

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        preds, _ = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(xb)
        correct += ((preds > 0.5).float() == yb).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct = 0.0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            preds, _ = model(xb)
            loss = criterion(preds, yb)
            total_loss += loss.item() * len(xb)
            correct += ((preds > 0.5).float() == yb).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

EPOCHS = 20
train_losses, val_losses = [], []

for epoch in range(EPOCHS):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion)
    vl_loss, vl_acc = eval_epoch(model, val_loader, criterion)
    train_losses.append(tr_loss)
    val_losses.append(vl_loss)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:02d} | "
              f"Train Loss: {tr_loss:.4f} Acc: {tr_acc:.3f} | "
              f"Val Loss: {vl_loss:.4f} Acc: {vl_acc:.3f}")


## Plot Training Curves

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses,   label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("BCE Loss")
plt.title("Behavioral Transformer — Training Curve")
plt.legend()
plt.tight_layout()
plt.savefig("transformer_training.png", dpi=150)
plt.show()

## Save Checkpoint

In [ ]:
torch.save(model.state_dict(), CHECKPOINT_DIR / "transformer.pt")
print(f"✅ Transformer saved → {CHECKPOINT_DIR / 'transformer.pt'}")

## Train GAN + Autoencoder (KYC)

In [ ]:
print("\nGenerating KYC data...")
doc_gen  = DocumentMetadataGenerator(seed=42)
forger   = ForgerySimulator(seed=42)

legit_docs  = doc_gen.generate_batch(n=2000)
forged_docs = forger.generate_forged_batch(legit_docs, n=400)
all_docs    = legit_docs + forged_docs

doc_features = np.array([doc_gen.to_feature_vector(d) for d in all_docs])
doc_labels   = np.array([d.label for d in all_docs], dtype=np.float32)

X_kyc = torch.tensor(doc_features, dtype=torch.float32)
y_kyc = torch.tensor(doc_labels,   dtype=torch.float32).unsqueeze(1)

kyc_dataset  = TensorDataset(X_kyc, y_kyc)
kyc_n_train  = int(0.8 * len(kyc_dataset))
kyc_n_val    = len(kyc_dataset) - kyc_n_train
kyc_train_ds, kyc_val_ds = random_split(kyc_dataset, [kyc_n_train, kyc_n_val])

kyc_train_loader = DataLoader(kyc_train_ds, batch_size=32, shuffle=True)

kyc_model = GANAutoencoderKYC().to(DEVICE)
kyc_optim = optim.Adam(kyc_model.parameters(), lr=1e-4)
kyc_criterion = nn.BCELoss()

kyc_losses = []
for epoch in range(20):
    kyc_model.train()
    epoch_loss = 0.0
    for xb, yb in kyc_train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        kyc_optim.zero_grad()
        preds = kyc_model(xb)
        loss  = kyc_criterion(preds, yb)
        loss.backward()
        kyc_optim.step()
        epoch_loss += loss.item() * len(xb)
    kyc_losses.append(epoch_loss / len(kyc_train_loader.dataset))
    if (epoch + 1) % 5 == 0:
        print(f"KYC Epoch {epoch+1:02d} | Loss: {kyc_losses[-1]:.4f}")

torch.save(kyc_model.state_dict(), CHECKPOINT_DIR / "gan_autoencoder.pt")
print(f"✅ GAN + Autoencoder saved → {CHECKPOINT_DIR / 'gan_autoencoder.pt'}")


## Evaluation Summary

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        preds, _ = model(xb.to(DEVICE))
        all_preds.extend(preds.cpu().numpy().flatten())
        all_labels.extend(yb.numpy().flatten())

binary_preds = [1 if p > 0.5 else 0 for p in all_preds]
auc = roc_auc_score(all_labels, all_preds)
print(f"\nTransformer AUC-ROC: {auc:.4f}")
print(classification_report(all_labels, binary_preds, target_names=["Legit", "Fraud"]))
